# NBA Win Probability - Model Training

Predict the probability that the **home team wins** an NBA regular-season game
from pre-game features only (Elo, rest, recent form, head-to-head history).

**Data.** One row per game, built by `src/feature_engineering.py` →
`data/processed/games_final.csv`.

**Split.** Chronological, by season - no shuffling, so every model is scored on
games played strictly *after* the ones it was trained on:

| split | seasons | purpose |
|-------|---------|---------|
| train | 2020-21 … 2023-24 | fit model parameters |
| val   | 2024-25 | pick hyper-parameters / model |
| test  | 2025-26 | reported once, at the end |

**Metrics.** `log_loss` is the headline number (this is a probability model);
`accuracy` (@ 0.5) and `AUC` are secondary. `brier` is tracked for calibration.

In [1]:
import sys, os, warnings
from itertools import product

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, brier_score_loss, mean_squared_error
from xgboost import XGBClassifier, XGBRegressor

from src.train_model import load_and_split_data, split_by_season

RANDOM_STATE = 42
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## Data & split

In [2]:
X, y, y_margin, metadata = load_and_split_data("../data/processed/games_final.csv")

TRAIN_SEASONS = [22020, 22021, 22022, 22023]
VAL_SEASONS   = [22024]
TEST_SEASONS  = [22025]

X_train, X_val, X_test, y_train, y_val, y_test, meta_train, meta_val, meta_test = split_by_season(
    X, y, metadata, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
)

# same split for the margin regression target (home points - away points)
m_train, m_val, m_test = (y_margin.loc[mt.index] for mt in (meta_train, meta_val, meta_test))

print(f"Train / Val / Test : {len(X_train)} / {len(X_val)} / {len(X_test)} games")
print(f"Features           : {X_train.shape[1]}")
print(f"Home-win rate      : train {y_train.mean():.3f} | val {y_val.mean():.3f} | test {y_test.mean():.3f}")

Train: 4770 matches ([22020, 22021, 22022, 22023])
Val:   1230 matches ([22024])
Test:  1230 matches ([22025])
Train / Val / Test : 4770 / 1230 / 1230 games
Features           : 111
Home-win rate      : train 0.553 | val 0.544 | test 0.554


## Feature overview

The features group into Elo ratings (with a margin-of-victory multiplier),
schedule load (rest, back-to-back, games in the last 3 / 7 days), recent form
(streak plus rolling box-score and possession-efficiency averages over 5 and 10
games, split by home / road venue), head-to-head history, and season-context
flags. Early-season games have missing form values by construction - left as
`NaN` (XGBoost handles them natively; the logistic-regression path imputes with
the train median).

In [3]:
feature_groups = {
    "elo":        [c for c in X.columns if "elo" in c],
    "rest":       [c for c in X.columns if "rest" in c or "back_to_back" in c],
    "schedule":   [c for c in X.columns if "last_7d" in c or "last_3d" in c or "_in_four" in c or "_in_six" in c],
    "form":       [c for c in X.columns if ("streak" in c or "rolling" in c) and "rating" not in c and "poss" not in c],
    "efficiency": [c for c in X.columns if "rating" in c or "poss" in c],
    "matchup":    [c for c in X.columns if "matchup" in c],
    "context":    [c for c in X.columns if "games_played" in c or "early_season" in c],
}
for name, cols in feature_groups.items():
    print(f"{name:11s} {len(cols):2d} features")

missing = X_train.isna().mean()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nColumns with missing values in train: {len(missing)}")
print(missing.round(3).head(25).to_string())

elo          4 features
rest         5 features
schedule    10 features
form        36 features
efficiency  36 features
matchup     18 features
context      3 features

Columns with missing values in train: 87
HOME_matchup_pts_5                 0.0910
AWAY_matchup_pts_5                 0.0910
HOME_matchup_reb_5                 0.0910
AWAY_matchup_reb_5                 0.0910
HOME_matchup_ast_5                 0.0910
AWAY_matchup_ast_5                 0.0910
HOME_matchup_stl_5                 0.0910
AWAY_matchup_stl_5                 0.0910
HOME_matchup_blk_5                 0.0910
AWAY_matchup_blk_5                 0.0910
HOME_matchup_tov_5                 0.0910
AWAY_matchup_tov_5                 0.0910
HOME_matchup_fg_pct_5              0.0910
AWAY_matchup_fg_pct_5              0.0910
HOME_matchup_fg3_pct_5             0.0910
AWAY_matchup_fg3_pct_5             0.0910
venue_net_rating_diff_10           0.0320
venue_net_rating_diff_5            0.0320
HOME_rolling_venue_net_rating_5   

## Metrics helper

In [4]:
results = []

def evaluate(name, split, y_true, proba):
    row = {
        "model": name,
        "split": split,
        "accuracy": accuracy_score(y_true, (proba >= 0.5).astype(int)),
        "log_loss": log_loss(y_true, proba),
        "auc": roc_auc_score(y_true, proba),
        "brier": brier_score_loss(y_true, proba),
    }
    results.append(row)
    print(f"{name:20s} [{split:4s}]  acc={row['accuracy']:.3f}   "
          f"log_loss={row['log_loss']:.4f}   auc={row['auc']:.3f}")
    return row

## 1. Baseline — home team always wins

The home team wins ~55% of games. Any useful model must beat this on log loss,
not just accuracy.

In [5]:
proba_base_val = np.full(len(y_val), y_train.mean())
evaluate("baseline (home)", "val", y_val, proba_base_val)

baseline (home)      [val ]  acc=0.544   log_loss=0.6895   auc=0.500


{'model': 'baseline (home)',
 'split': 'val',
 'accuracy': 0.5439024390243903,
 'log_loss': 0.6894560371280906,
 'auc': 0.5,
 'brier': 0.24815606780331087}

## 2. Logistic Regression

Missing values imputed with the train median, features standardised, L2 penalty.
The regularisation strength `C` is swept on the validation set.

In [6]:
train_median = X_train.median()
Xtr_f, Xval_f, Xte_f = (df.fillna(train_median) for df in (X_train, X_val, X_test))

scaler = StandardScaler().fit(Xtr_f)
Xtr_s, Xval_s, Xte_s = (scaler.transform(df) for df in (Xtr_f, Xval_f, Xte_f))

print("Regularisation sweep (val):")
for C in [0.003, 0.01, 0.02, 0.03, 0.1, 0.3, 1.0]:
    lr = LogisticRegression(max_iter=2000, C=C).fit(Xtr_s, y_train)
    p = lr.predict_proba(Xval_s)[:, 1]
    print(f"  C={C:<6} log_loss={log_loss(y_val, p):.4f}  acc={accuracy_score(y_val, p >= 0.5):.4f}")

BEST_C = 0.02
logreg = LogisticRegression(max_iter=2000, C=BEST_C).fit(Xtr_s, y_train)
proba_lr_val = logreg.predict_proba(Xval_s)[:, 1]
evaluate(f"logreg (C={BEST_C})", "val", y_val, proba_lr_val)

Regularisation sweep (val):


  C=0.003  log_loss=0.6070  acc=0.6593
  C=0.01   log_loss=0.6082  acc=0.6659
  C=0.02   log_loss=0.6092  acc=0.6650


  C=0.03   log_loss=0.6097  acc=0.6634
  C=0.1    log_loss=0.6109  acc=0.6618


  C=0.3    log_loss=0.6118  acc=0.6602


  C=1.0    log_loss=0.6127  acc=0.6553
logreg (C=0.02)      [val ]  acc=0.665   log_loss=0.6092   auc=0.721


{'model': 'logreg (C=0.02)',
 'split': 'val',
 'accuracy': 0.6650406504065041,
 'log_loss': 0.6091793349476846,
 'auc': 0.7208273715791521,
 'brier': 0.2111403492095447}

## 3. XGBoost

`XGBClassifier` with native `NaN` handling (no imputation) and **early stopping**
on the validation set, so the tree count is chosen automatically — never
hand-tuned.

The grid below deliberately searches *small* models. With only ~4.8k training
games and a signal that is mostly linear (Elo difference dominates), deep trees
memorise the training set and lose on validation. Shallow trees + a slow learning
rate + row/column subsampling is the regime that generalises.

In [7]:
def fit_xgb(params, n_estimators=3000, patience=50):
    model = XGBClassifier(
        n_estimators=n_estimators,
        eval_metric="logloss",
        early_stopping_rounds=patience,
        random_state=RANDOM_STATE,
        **params,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

grid = {
    "max_depth":        [2, 3, 4],
    "learning_rate":    [0.02, 0.05],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

rows = []
for combo in product(*grid.values()):
    params = dict(zip(grid, combo), min_child_weight=1, reg_lambda=1.0)
    m = fit_xgb(params)
    p_val = m.predict_proba(X_val)[:, 1]
    rows.append({
        **{k: params[k] for k in grid},
        "best_iter": m.best_iteration,
        "train_ll":  log_loss(y_train, m.predict_proba(X_train)[:, 1]),
        "val_ll":    log_loss(y_val, p_val),
        "val_acc":   accuracy_score(y_val, p_val >= 0.5),
    })

grid_df = pd.DataFrame(rows).sort_values("val_ll").reset_index(drop=True)
print("Grid search - best 10 by validation log loss.")
print("Depth-2 models dominate; deeper trees cut train_ll but raise val_ll (overfitting):")
grid_df.head(10)

Grid search - best 10 by validation log loss.
Depth-2 models dominate; deeper trees cut train_ll but raise val_ll (overfitting):


,max_depth,learning_rate,subsample,colsample_bytree,best_iter,train_ll,val_ll,val_acc
0,3,0.0200,0.9000,0.9000,204,0.5859,0.6104,0.6553
1,3,0.0200,0.9000,0.7000,205,0.5869,0.6105,0.6618
2,2,0.0500,0.9000,0.7000,111,0.6064,0.6107,0.6634
3,3,0.0500,0.9000,0.7000,117,0.5705,0.6107,0.6626
4,2,0.0200,0.7000,0.7000,284,0.6053,0.6109,0.6642
5,2,0.0500,0.7000,0.7000,72,0.6161,0.6110,0.6610
6,2,0.0500,0.7000,0.9000,128,0.6013,0.6111,0.6585
7,4,0.0500,0.9000,0.9000,59,0.5615,0.6115,0.6569
8,2,0.0200,0.9000,0.7000,283,0.6066,0.6116,0.6618
9,2,0.0200,0.9000,0.9000,285,0.6057,0.6116,0.6593


### Chosen configuration

| parameter | value | why |
|-----------|-------|-----|
| `max_depth` | **2** | shallow interactions only; depth ≥ 4 overfits (val log loss +0.008) |
| `learning_rate` | **0.02** | slow shrinkage; pairs with early stopping (~400–500 trees) |
| `n_estimators` | **3000 cap** | real count set by `early_stopping_rounds=50` on val |
| `subsample` | **0.7** | row bagging — decorrelates trees on a small dataset |
| `colsample_bytree` | **0.7** | column bagging — dilutes the ~50 near-noise features |
| `min_child_weight` | **1** | little effect here; kept at default |
| `reg_lambda` / `reg_alpha` | **1.0 / 0.0** | mild L2; heavier L1 didn't help on val |

In [8]:
XGB_PARAMS = dict(
    max_depth=2,
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=1,
    reg_lambda=1.0,
    reg_alpha=0.0,
)

xgb = fit_xgb(XGB_PARAMS)
XGB_BEST_ITER = int(xgb.best_iteration)
proba_xgb_val = xgb.predict_proba(X_val)[:, 1]

print(f"Trees kept by early stopping : {xgb.best_iteration}")
print(f"Train log loss               : {log_loss(y_train, xgb.predict_proba(X_train)[:, 1]):.4f}")
print(f"Val   log loss               : {log_loss(y_val, proba_xgb_val):.4f}   "
      f"(small gap = not overfit)\n")
evaluate("xgboost", "val", y_val, proba_xgb_val)

Trees kept by early stopping : 284
Train log loss               : 0.6053
Val   log loss               : 0.6109   (small gap = not overfit)

xgboost              [val ]  acc=0.664   log_loss=0.6109   auc=0.723


{'model': 'xgboost',
 'split': 'val',
 'accuracy': 0.6642276422764227,
 'log_loss': 0.6108653545379639,
 'auc': 0.7234012507027544,
 'brier': 0.2113681137561798}

### Feature importance

`elo_diff` and `elo_win_prob` still dominate, but the possession-based
efficiency diffs (`venue_net_rating_diff`, `off_rating_diff`, `net_rating_adj_diff`)
and the schedule-congestion features now sit right below them instead of a flat
noise tail - the feature selection step (section 6) trims the rest.

In [9]:
importance = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 15 features:")
print(importance.head(15).round(4).to_string())
print("\nBottom 10 features:")
print(importance.tail(10).round(4).to_string())

Top 15 features:
elo_diff                           0.0545
elo_win_prob                       0.0461
venue_net_rating_diff_10           0.0192
HOME_elo                           0.0179
HOME_rolling_venue_net_rating_10   0.0137
venue_net_rating_diff_5            0.0128
HOME_rolling_off_rating_10         0.0125
AWAY_games_last_3d                 0.0119
games_last_3d_diff                 0.0117
AWAY_three_in_four                 0.0109
HOME_games_played_this_season      0.0108
off_rating_diff_10                 0.0105
HOME_matchup_tov_5                 0.0103
AWAY_elo                           0.0102
def_rating_diff_5                  0.0102

Bottom 10 features:
HOME_matchup_stl_5      0.0060
HOME_rolling_pts_5      0.0059
AWAY_rolling_poss_10    0.0050
HOME_games_last_3d      0.0000
HOME_four_in_six        0.0000
HOME_three_in_four      0.0000
matchup_history_count   0.0000
HOME_games_last_7d      0.0000
AWAY_four_in_six        0.0000
is_early_season         0.0000


## 4. Ensemble — logistic regression + XGBoost

The two models make different kinds of errors (linear vs. piece-wise constant),
so averaging their probabilities is a cheap way to improve ranking (AUC) and
match the best single-model log loss. Weight swept on val.

In [10]:
print("Blend weight sweep (val):")
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = w * proba_lr_val + (1 - w) * proba_xgb_val
    print(f"  {w:.1f}*logreg + {1 - w:.1f}*xgb   "
          f"log_loss={log_loss(y_val, p):.4f}   auc={roc_auc_score(y_val, p):.4f}")

BLEND_W = 0.5
proba_blend_val = BLEND_W * proba_lr_val + (1 - BLEND_W) * proba_xgb_val
evaluate("blend", "val", y_val, proba_blend_val)

Blend weight sweep (val):
  0.3*logreg + 0.7*xgb   log_loss=0.6089   auc=0.7252
  0.4*logreg + 0.6*xgb   log_loss=0.6085   auc=0.7253
  0.5*logreg + 0.5*xgb   log_loss=0.6083   auc=0.7251
  0.6*logreg + 0.4*xgb   log_loss=0.6081   auc=0.7246
  0.7*logreg + 0.3*xgb   log_loss=0.6082   auc=0.7238
blend                [val ]  acc=0.660   log_loss=0.6083   auc=0.725


{'model': 'blend',
 'split': 'val',
 'accuracy': 0.6601626016260163,
 'log_loss': 0.6082523753228712,
 'auc': 0.7251011832916343,
 'brier': 0.21042835272905802}

## 5. Rolling-origin cross-validation

One validation season is noisy - log-loss differences below ~0.005 are usually
within sampling error. Every model choice below (regularisation, feature set,
blend weight) is scored across three expanding-window folds instead of a single
`val` season, and the final test metrics get a bootstrap confidence interval.

In [11]:
CV_FOLDS = [
    ([22020, 22021],               22022),
    ([22020, 22021, 22022],        22023),
    ([22020, 22021, 22022, 22023], 22024),
]

def cv_score(fit_predict, folds=CV_FOLDS, features=None):
    cols = list(features) if features is not None else list(X.columns)
    rows = []
    for tr_seasons, va_season in folds:
        tr = metadata["SEASON_ID"].isin(tr_seasons).values
        va = (metadata["SEASON_ID"] == va_season).values
        p = np.clip(fit_predict(X.loc[tr, cols], y[tr].values, X.loc[va, cols]), 1e-6, 1 - 1e-6)
        yv = y[va].values
        rows.append((log_loss(yv, p), roc_auc_score(yv, p), brier_score_loss(yv, p)))
    a = np.array(rows)
    return {"log_loss": a[:, 0].mean(), "log_loss_std": a[:, 0].std(),
            "auc": a[:, 1].mean(), "brier": a[:, 2].mean()}

def bootstrap_ci(y_true, proba, metric, n=2000, seed=RANDOM_STATE):
    y_true, proba = np.asarray(y_true), np.asarray(proba)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = [metric(y_true[b], proba[b]) for b in (rng.choice(idx, len(idx), True) for _ in range(n))]
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def fp_logreg(C=BEST_C):
    def f(Xtr, ytr, Xva):
        med = Xtr.median()
        sc = StandardScaler().fit(Xtr.fillna(med))
        m = LogisticRegression(max_iter=2000, C=C).fit(sc.transform(Xtr.fillna(med)), ytr)
        return m.predict_proba(sc.transform(Xva.fillna(med)))[:, 1]
    return f

def fp_xgb(params=None, n_estimators=None):
    params = params or XGB_PARAMS
    n_estimators = n_estimators or XGB_BEST_ITER
    def f(Xtr, ytr, Xva):
        m = XGBClassifier(n_estimators=n_estimators, eval_metric="logloss",
                          random_state=RANDOM_STATE, **params)
        m.fit(Xtr, ytr, verbose=False)
        return m.predict_proba(Xva)[:, 1]
    return f

def fp_blend(w=BLEND_W):
    lr, xg = fp_logreg(), fp_xgb()
    return lambda Xtr, ytr, Xva: w * lr(Xtr, ytr, Xva) + (1 - w) * xg(Xtr, ytr, Xva)

cv_full = pd.DataFrame({
    "logreg":  cv_score(fp_logreg()),
    "xgboost": cv_score(fp_xgb()),
    "blend":   cv_score(fp_blend()),
}).T
print("Rolling-origin CV - val folds on 2022, 2023, 2024:")
cv_full.round(4)

Rolling-origin CV - val folds on 2022, 2023, 2024:


,log_loss,log_loss_std,auc,brier
logreg,0.6274,0.0210,0.6914,0.2192
xgboost,0.6257,0.0174,0.6963,0.2181
blend,0.6240,0.0188,0.6972,0.2175


## 6. Feature selection

There are ~110 features but importance has a long flat tail. Two prunes are
combined: an L1-penalised logistic regression, and XGBoost gain + permutation
importance on `val`. The union is scored with rolling-origin CV against the full
model.

In [12]:
# --- L1 logistic-regression path -----------------------------------------------
med = X_train.median()
Xtr_imp = X_train.fillna(med)
sc_l1 = StandardScaler().fit(Xtr_imp)
for C in [0.005, 0.01, 0.02, 0.05, 0.1]:
    l1 = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000)
    l1.fit(sc_l1.transform(Xtr_imp), y_train)
    print(f"  L1 C={C:<6} non-zero coefs: {int((l1.coef_[0] != 0).sum())}")

L1_C = 0.05
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=L1_C, max_iter=2000).fit(
    sc_l1.transform(Xtr_imp), y_train)
l1_features = list(X.columns[l1.coef_[0] != 0])

# --- XGBoost gain + permutation importance -----------------------------------
gain = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
perm = permutation_importance(xgb, X_val, y_val, n_repeats=10,
                              random_state=RANDOM_STATE, scoring="neg_log_loss")
perm = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
xgb_features = sorted(set(gain.head(12).index) | set(perm.head(12).index))

SELECTED = sorted(set(l1_features) | set(xgb_features))
print(f"\nL1 keeps {len(l1_features)}  |  XGB keeps {len(xgb_features)}  |  union = {len(SELECTED)}")
print(SELECTED)

cv_sel = pd.DataFrame({
    "logreg_full": cv_score(fp_logreg()),
    "logreg_sel":  cv_score(fp_logreg(), features=SELECTED),
    "xgb_full":    cv_score(fp_xgb()),
    "xgb_sel":     cv_score(fp_xgb(), features=SELECTED),
    "blend_sel":   cv_score(fp_blend(), features=SELECTED),
}).T
cv_sel.round(4)

  L1 C=0.005  non-zero coefs: 2
  L1 C=0.01   non-zero coefs: 7
  L1 C=0.02   non-zero coefs: 13


  L1 C=0.05   non-zero coefs: 30


  L1 C=0.1    non-zero coefs: 49



L1 keeps 30  |  XGB keeps 19  |  union = 39
['AWAY_elo', 'AWAY_games_last_3d', 'AWAY_games_last_7d', 'AWAY_is_back_to_back', 'AWAY_matchup_fg3_pct_5', 'AWAY_matchup_pts_5', 'AWAY_matchup_reb_5', 'AWAY_matchup_stl_5', 'AWAY_rest_days', 'AWAY_rolling_stl_5', 'AWAY_rolling_tov_10', 'AWAY_rolling_tov_5', 'AWAY_three_in_four', 'HOME_elo', 'HOME_games_last_3d', 'HOME_games_played_this_season', 'HOME_is_back_to_back', 'HOME_matchup_ast_5', 'HOME_matchup_blk_5', 'HOME_matchup_fg3_pct_5', 'HOME_matchup_reb_5', 'HOME_rest_days', 'HOME_rolling_blk_10', 'HOME_rolling_off_rating_10', 'HOME_rolling_poss_10', 'HOME_rolling_stl_10', 'HOME_rolling_tov_5', 'HOME_rolling_venue_net_rating_10', 'def_rating_diff_10', 'elo_diff', 'elo_win_prob', 'games_last_3d_diff', 'matchup_history_count', 'net_rating_adj_diff_10', 'off_rating_diff_10', 'poss_diff_10', 'rest_days_diff', 'venue_net_rating_diff_10', 'venue_net_rating_diff_5']


,log_loss,log_loss_std,auc,brier
logreg_full,0.6274,0.0210,0.6914,0.2192
logreg_sel,0.6212,0.0178,0.7010,0.2162
xgb_full,0.6257,0.0174,0.6963,0.2181
xgb_sel,0.6232,0.0158,0.7004,0.2169
blend_sel,0.6206,0.0168,0.7029,0.2159


## 7. Margin model

Predict the point margin (home - away) with a regressor, then convert to a win
probability. A margin model is often better calibrated than direct
classification and can be sanity-checked against a market line. Same "small
model" regime as the classifier; margin -> probability link (normal CDF vs.
logistic) picked on `val`.

In [13]:
xgb_m = XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=2,
                     subsample=0.7, colsample_bytree=0.7, min_child_weight=1,
                     reg_lambda=1.0, eval_metric="rmse", early_stopping_rounds=50,
                     random_state=RANDOM_STATE)
xgb_m.fit(X_train, m_train, eval_set=[(X_val, m_val)], verbose=False)
ridge_m = Ridge(alpha=10.0).fit(Xtr_s, m_train)   # Xtr_s: standardised+imputed, from the logreg cell

pm_train, pm_val, pm_test = (xgb_m.predict(d) for d in (X_train, X_val, X_test))
rmse_xgb = mean_squared_error(m_val, pm_val) ** 0.5
rmse_ridge = mean_squared_error(m_val, ridge_m.predict(Xval_s)) ** 0.5
print(f"margin val RMSE : xgb {rmse_xgb:.2f}  |  ridge {rmse_ridge:.2f}")

# link A: normal CDF with train residual std ; link B: logistic on predicted margin
resid_std = np.std(m_train.values - pm_train)
pmv_a, pmt_a = norm.cdf(pm_val / resid_std), norm.cdf(pm_test / resid_std)
link = LogisticRegression().fit(pm_train.reshape(-1, 1), y_train)
pmv_b = link.predict_proba(pm_val.reshape(-1, 1))[:, 1]
pmt_b = link.predict_proba(pm_test.reshape(-1, 1))[:, 1]

if log_loss(y_val, pmv_b) < log_loss(y_val, pmv_a):
    MARGIN_LINK, proba_margin_val, proba_margin_test = "logit", pmv_b, pmt_b
else:
    MARGIN_LINK, proba_margin_val, proba_margin_test = "normcdf", pmv_a, pmt_a
print(f"margin->prob link: {MARGIN_LINK}")
evaluate("margin (xgb)", "val", y_val, proba_margin_val)

margin val RMSE : xgb 14.16  |  ridge 14.26
margin->prob link: logit
margin (xgb)         [val ]  acc=0.654   log_loss=0.6103   auc=0.723


{'model': 'margin (xgb)',
 'split': 'val',
 'accuracy': 0.6536585365853659,
 'log_loss': 0.6102967858314514,
 'auc': 0.7234065796450391,
 'brier': 0.2113560140132904}

## 8. Final evaluation - held-out test season (2025-26)

Run once. Model choice, hyper-parameters, feature subset, margin link and blend
weight were all fixed on the validation season / CV folds above.

In [14]:
proba_base_test  = np.full(len(y_test), y_train.mean())
proba_lr_test    = logreg.predict_proba(Xte_s)[:, 1]
proba_xgb_test   = xgb.predict_proba(X_test)[:, 1]
proba_blend_test = BLEND_W * proba_lr_test + (1 - BLEND_W) * proba_xgb_test

# blend refit on the selected feature subset only
_blend = fp_blend()
proba_sel_val  = _blend(X_train[SELECTED], y_train.values, X_val[SELECTED])
proba_sel_test = _blend(X_train[SELECTED], y_train.values, X_test[SELECTED])
evaluate("blend (selected)", "val", y_val, proba_sel_val)

evaluate("baseline (home)",     "test", y_test, proba_base_test)
evaluate(f"logreg (C={BEST_C})", "test", y_test, proba_lr_test)
evaluate("xgboost",             "test", y_test, proba_xgb_test)
evaluate("blend",              "test", y_test, proba_blend_test)
evaluate("blend (selected)",   "test", y_test, proba_sel_test)
evaluate("margin (xgb)",       "test", y_test, proba_margin_test)

blend (selected)     [val ]  acc=0.664   log_loss=0.6068   auc=0.727
baseline (home)      [test]  acc=0.554   log_loss=0.6872   auc=0.500
logreg (C=0.02)      [test]  acc=0.674   log_loss=0.6053   auc=0.726
xgboost              [test]  acc=0.685   log_loss=0.6025   auc=0.733
blend                [test]  acc=0.678   log_loss=0.6019   auc=0.732
blend (selected)     [test]  acc=0.683   log_loss=0.6018   auc=0.732
margin (xgb)         [test]  acc=0.680   log_loss=0.5985   auc=0.735


{'model': 'margin (xgb)',
 'split': 'test',
 'accuracy': 0.6796747967479675,
 'log_loss': 0.5985127687454224,
 'auc': 0.7347271871053365,
 'brier': 0.20586776733398438}

### Probability calibration

The deliverable is a win probability, so it should match observed frequencies.
Platt (logistic on the log-odds) and isotonic are fit on the `val` season and
applied to `test`; the better one on `val` log loss is kept.

In [15]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def make_cals(p_fit, y_fit):
    pl = LogisticRegression().fit(logit(p_fit).reshape(-1, 1), y_fit)
    iso_ = IsotonicRegression(out_of_bounds="clip").fit(p_fit, y_fit)
    return {
        "raw":      lambda p: np.asarray(p),
        "platt":    lambda p: pl.predict_proba(logit(p).reshape(-1, 1))[:, 1],
        "isotonic": lambda p: iso_.predict(p),
    }

# isotonic overfits ~1k points, so the calibrator is CHOSEN on a held-out slice
# of val (chronological 60/40), then refit on the full val season.
yv = y_val.to_numpy()
cut = int(len(yv) * 0.6)
pick = make_cals(proba_blend_val[:cut], yv[:cut])
held = {k: log_loss(yv[cut:], f(proba_blend_val[cut:])) for k, f in pick.items()}
CAL_METHOD = min(held, key=held.get)
print("held-out val log loss by calibrator:", {k: round(v, 4) for k, v in held.items()})
print("chosen:", CAL_METHOD)

final_cal = make_cals(proba_blend_val, yv)[CAL_METHOD]
proba_cal_val  = final_cal(proba_blend_val)
proba_cal_test = final_cal(proba_blend_test)

cal_tbl = pd.DataFrame({
    name: {"val_log_loss": log_loss(y_val, pv),  "val_brier": brier_score_loss(y_val, pv),
           "test_log_loss": log_loss(y_test, pt), "test_brier": brier_score_loss(y_test, pt)}
    for name, pv, pt in [
        ("raw",              proba_blend_val, proba_blend_test),
        (f"cal ({CAL_METHOD})", proba_cal_val, proba_cal_test)]
}).T
print(cal_tbl.round(4))

evaluate(f"blend+{CAL_METHOD}", "val",  y_val,  proba_cal_val)
evaluate(f"blend+{CAL_METHOD}", "test", y_test, proba_cal_test)

held-out val log loss by calibrator: {'raw': 0.581, 'platt': 0.5819, 'isotonic': 0.5823}
chosen: raw
           val_log_loss  val_brier  test_log_loss  test_brier
raw              0.6083     0.2104         0.6019      0.2071
cal (raw)        0.6083     0.2104         0.6019      0.2071
blend+raw            [val ]  acc=0.660   log_loss=0.6083   auc=0.725
blend+raw            [test]  acc=0.678   log_loss=0.6019   auc=0.732


{'model': 'blend+raw',
 'split': 'test',
 'accuracy': 0.6780487804878049,
 'log_loss': 0.6018734122333973,
 'auc': 0.7320702313932829,
 'brier': 0.2071142328622252}

### Bootstrap confidence intervals (test season)

2000 resamples of the 2025-26 test games. Overlapping intervals = the models are
statistically indistinguishable on this sample.

In [16]:
print("Bootstrap 95% CI on test (2000 resamples):")
for name, p in [("logreg", proba_lr_test), ("xgboost", proba_xgb_test),
                ("blend", proba_blend_test), ("blend (selected)", proba_sel_test),
                ("margin (xgb)", proba_margin_test), (f"blend+{CAL_METHOD}", proba_cal_test)]:
    ll = bootstrap_ci(y_test, p, log_loss)
    au = bootstrap_ci(y_test, p, roc_auc_score)
    print(f"  {name:18s} log_loss {ll[0]:.4f} [{ll[1]:.4f}, {ll[2]:.4f}]   "
          f"auc {au[0]:.3f} [{au[1]:.3f}, {au[2]:.3f}]")

Bootstrap 95% CI on test (2000 resamples):


  logreg             log_loss 0.6051 [0.5838, 0.6258]   auc 0.726 [0.699, 0.755]


  xgboost            log_loss 0.6023 [0.5829, 0.6210]   auc 0.733 [0.705, 0.762]


  blend              log_loss 0.6017 [0.5815, 0.6213]   auc 0.732 [0.704, 0.761]


  blend (selected)   log_loss 0.6016 [0.5810, 0.6207]   auc 0.732 [0.705, 0.761]


  margin (xgb)       log_loss 0.5983 [0.5746, 0.6200]   auc 0.735 [0.708, 0.763]


  blend+raw          log_loss 0.6017 [0.5815, 0.6213]   auc 0.732 [0.704, 0.761]


## Results summary

In [17]:
summary = pd.DataFrame(results)[["model", "split", "accuracy", "log_loss", "auc", "brier"]]
display(summary)

print("\nLog loss by split:")
display(pd.DataFrame(results).pivot(index="model", columns="split", values="log_loss")
        .reindex(columns=["val", "test"]))

,model,split,accuracy,log_loss,auc,brier
0,baseline (home),val,0.5439,0.6895,0.5000,0.2482
1,logreg (C=0.02),val,0.6650,0.6092,0.7208,0.2111
2,xgboost,val,0.6642,0.6109,0.7234,0.2114
3,blend,val,0.6602,0.6083,0.7251,0.2104
4,margin (xgb),val,0.6537,0.6103,0.7234,0.2114
5,blend (selected),val,0.6642,0.6068,0.7269,0.2097
6,baseline (home),test,0.5545,0.6872,0.5000,0.2470
7,logreg (C=0.02),test,0.6740,0.6053,0.7261,0.2087
8,xgboost,test,0.6854,0.6025,0.7329,0.2073
9,blend,test,0.6780,0.6019,0.7321,0.2071



Log loss by split:


split,val,test
model,,
baseline (home),0.6895,0.6872
blend,0.6083,0.6019
blend (selected),0.6068,0.6018
blend+raw,0.6083,0.6019
logreg (C=0.02),0.6092,0.6053
margin (xgb),0.6103,0.5985
xgboost,0.6109,0.6025


### Calibration check (validation)

Because the deliverable is a *win probability*, the predicted probability should
match the observed win rate inside each bucket. `pred` ≈ `actual` per row = well
calibrated.

In [18]:
def calibration_table(y_true, proba, bins=10):
    df = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(proba)})
    df["bucket"] = pd.qcut(df["p"], bins, duplicates="drop")
    return (df.groupby("bucket", observed=True)
              .agg(n=("y", "size"), pred=("p", "mean"), actual=("y", "mean"))
              .round(3))

print("Blended model, validation season:")
calibration_table(y_val, proba_blend_val)

Blended model, validation season:


,n,pred,actual
bucket,,,
"(0.155, 0.293]",123,0.2430,0.1710
"(0.293, 0.373]",123,0.3340,0.3660
"(0.373, 0.448]",123,0.4150,0.3900
"(0.448, 0.52]",123,0.4830,0.4800
"(0.52, 0.57]",123,0.5460,0.5370
"(0.57, 0.621]",123,0.5970,0.5280
"(0.621, 0.676]",123,0.6490,0.6500
"(0.676, 0.736]",123,0.7070,0.6830
"(0.736, 0.798]",123,0.7680,0.7890


## Conclusions

### Where the model stands

| model | val log loss | test log loss | test AUC | test Brier |
|-------|-------------:|--------------:|---------:|-----------:|
| baseline (home) | 0.690 | 0.687 | 0.500 | 0.247 |
| logreg (C=0.02) | 0.609 | 0.605 | 0.726 | 0.209 |
| xgboost | 0.611 | 0.603 | 0.733 | 0.207 |
| blend (50/50) | 0.608 | 0.602 | 0.732 | 0.207 |
| blend, 39 selected features | 0.607 | 0.602 | 0.732 | 0.207 |
| margin model (XGB regressor) | 0.610 | **0.599** | **0.735** | **0.206** |

Best model ~0.60 log loss / ~0.67 accuracy / ~0.73 AUC vs. an always-home
baseline of ~0.69. The gap is real but modest - from pre-game information alone,
NBA outcomes sit close to the usual ceiling for no-odds models. `elo_diff` and
`elo_win_prob` carry most of the signal; the possession-efficiency and schedule
features are the next tier, and everything below that is a flat tail.

Bootstrap 95% CIs on the 1230 test games are ~+/-0.02 log loss and every top
model's CI overlaps, so the ranking between blend / selected-blend / margin is
not statistically resolved on a single season.

### Feature groups

* **Elo** (`elo_diff`, `elo_win_prob`, `HOME/AWAY_elo`) - rating with a
  margin-of-victory multiplier; `k`, home advantage and season regression tuned
  on a rolling-origin split.
* **Possession efficiency** - offensive / defensive / net rating per 100
  possessions (rolling 5 & 10), an opponent-Elo-adjusted net rating, and a pace
  estimate.
* **Venue form** - the home team is scored on its home net rating, the away team
  on its road net rating; `venue_net_rating_diff_10` is the strongest non-Elo
  feature.
* **Schedule load** - games in the last 3 / 7 days, 3-in-4 and 4-in-6 flags.
* **Head-to-head** - last 5 meetings of the pair, cross-season.

### Model choice

The 50/50 **blend on the 39 selected features** is the recommended model: best
rolling-origin CV log loss, best AUC / Brier, and far fewer inputs than the full
set for the same score. The raw blend is already well calibrated (a held-out
calibrator check on `val` prefers no adjustment), so no Platt / isotonic step is
applied. The margin model is kept as a secondary probability estimate and a
sanity check against a market line.

For live predictions on future seasons, refit the chosen model on train + val
combined - `val` is only held out for model selection.

### Future work

* **Opponent-adjusted efficiency** via ridge / SRS (solve for team offensive and
  defensive strength jointly) rather than the current Elo-based correction.
* **Joint tuning** of the Elo margin multiplier and `k`; an Elo variant that
  updates directly on point margin.
* **More history** - pre-2020 seasons would tighten the confidence intervals
  enough to choose between the top models.
* **Stacking** - a meta-model on out-of-fold predictions instead of a fixed
  50/50 average.
* **Player availability** (injury reports / projected minutes) is the largest
  missing input and needs a new data source.